# V2 Phase 7 — Colab GPU smoke (Qwen3-8B)

**Strategy:** standard Google Colab notebook + GPU runtime (not Colab CLI).

Preserves: Qwen3-8B, `src/models` abstraction, fingerprinting, checkpoint/resume design.

**Before running:** Runtime → Change runtime type → **GPU**.

## 1. Auto-detect V2 (with Google Drive mount)

Set options in the next cell, then run it.

| Option | Meaning |
| --- | --- |
| `MOUNT_DRIVE = True` | Mount Google Drive, then search under MyDrive for V2 |
| `MANUAL_V2_ROOT` | Force a path if you already know it |
| `ALLOW_ZIP_UPLOAD = True` | Last-resort zip upload (off by default) |

A valid V2 root contains `config/experiment.yaml` and `scripts/smoke_generate.py`.

**Tip:** Put/copy the `V2` folder into Google Drive (or the whole CAPSTONE project), set `MOUNT_DRIVE = True`, re-run.

In [6]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import zipfile

# =============================================================================
# USER OPTIONS — edit these, then re-run this cell
# =============================================================================
MOUNT_DRIVE = True  # mount Google Drive, then auto-search for V2
# If you know the exact path after mounting, set it here (leave "" to auto-detect):
MANUAL_V2_ROOT = "/content/drive/MyDrive/.../V2"  # e.g. "/content/drive/MyDrive/CAPSTONE (RAG WITH UNCERTAINITY QUANTIFICATION)/V2"
DRIVE_SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
]
ALLOW_ZIP_UPLOAD = False  # set True only if Drive mount / auto-detect fails
MAX_SEARCH_DEPTH = 6
# =============================================================================


def looks_like_v2(path: Path) -> bool:
    try:
        return (path / "config" / "experiment.yaml").is_file() and (
            path / "scripts" / "smoke_generate.py"
        ).is_file()
    except OSError:
        return False


def _iter_dirs_limited(root: Path, max_depth: int):
    root = Path(root)
    if not root.is_dir():
        return
    yield root
    frontier = [(root, 0)]
    while frontier:
        current, depth = frontier.pop()
        if depth >= max_depth:
            continue
        try:
            children = sorted(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            name = child.name
            if name.startswith(".") or name in {
                "node_modules",
                ".git",
                "__pycache__",
                ".venv",
                "venv",
                "results",
                "knowledge_base",
            }:
                continue
            yield child
            frontier.append((child, depth + 1))


def discover_v2(extra_roots: list[Path] | None = None) -> Path | None:
    """Return first valid V2 root found (prefer shorter / explicit paths)."""
    seeds: list[Path] = [
        Path.cwd(),
        Path.cwd() / "V2",
        Path("/content/V2"),
        Path("/content"),
    ]
    if MANUAL_V2_ROOT.strip():
        seeds.insert(0, Path(MANUAL_V2_ROOT.strip()))
    if extra_roots:
        seeds.extend(extra_roots)

    candidates: list[Path] = []
    for seed in seeds:
        if looks_like_v2(seed):
            candidates.append(seed.resolve())
        v2_child = seed / "V2"
        if looks_like_v2(v2_child):
            candidates.append(v2_child.resolve())
        for d in _iter_dirs_limited(seed, MAX_SEARCH_DEPTH):
            if d.name == "V2" and looks_like_v2(d):
                candidates.append(d.resolve())
            elif looks_like_v2(d):
                candidates.append(d.resolve())

    uniq: list[Path] = []
    seen: set[str] = set()
    for c in candidates:
        key = str(c)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(c)
    if not uniq:
        return None
    uniq.sort(key=lambda p: (len(p.parts), str(p)))
    return uniq[0]


def mount_google_drive() -> None:
    from google.colab import drive

    mount_point = Path("/content/drive")
    if (mount_point / "MyDrive").is_dir():
        print("Google Drive already mounted at /content/drive")
        return
    print("Mounting Google Drive …")
    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted.")


# --- 1) Optional manual path ---
V2_ROOT: Path | None = None
if MANUAL_V2_ROOT.strip():
    manual = Path(MANUAL_V2_ROOT.strip())
    if looks_like_v2(manual):
        V2_ROOT = manual.resolve()
        print("Using MANUAL_V2_ROOT:", V2_ROOT)
    else:
        print("MANUAL_V2_ROOT set but invalid:", manual)

# --- 2) Local /content search (no Drive yet) ---
if V2_ROOT is None:
    V2_ROOT = discover_v2()
    if V2_ROOT is not None:
        print("Auto-detected V2 (before Drive):", V2_ROOT)

# --- 3) Mount Drive and search ---
if V2_ROOT is None and MOUNT_DRIVE:
    mount_google_drive()
    V2_ROOT = discover_v2(extra_roots=DRIVE_SEARCH_ROOTS)
    if V2_ROOT is not None:
        print("Auto-detected V2 on Drive:", V2_ROOT)
    else:
        print("Drive mounted, but no V2 found under:")
        for r in DRIVE_SEARCH_ROOTS:
            print(" -", r, "(exists:", r.is_dir(), ")")

# --- 4) Optional zip upload (off by default) ---
if V2_ROOT is None and ALLOW_ZIP_UPLOAD:
    print("Upload a zip of the V2 folder …")
    from google.colab import files

    uploaded = files.upload()
    zip_name = next(iter(uploaded), None)
    if not zip_name:
        raise FileNotFoundError("No zip uploaded.")
    zip_path = Path("/content") / zip_name
    extract_dir = Path("/content/V2_upload")
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    V2_ROOT = discover_v2(extra_roots=[extract_dir, Path("/content")])
    if V2_ROOT is None:
        hits = list(extract_dir.rglob("scripts/smoke_generate.py"))
        if hits:
            cand = hits[0].parents[1]
            if looks_like_v2(cand):
                V2_ROOT = cand.resolve()

if V2_ROOT is None or not looks_like_v2(V2_ROOT):
    raise FileNotFoundError(
        "V2 not found.\n"
        "Fix options at the top of this cell and re-run:\n"
        "  1) Set MOUNT_DRIVE = True (default), put V2 on Google Drive, re-run\n"
        "  2) Or set MANUAL_V2_ROOT = '/content/drive/MyDrive/.../V2'\n"
        "  3) Or set ALLOW_ZIP_UPLOAD = True and upload V2.zip"
    )

os.chdir(V2_ROOT)
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))
print("V2_ROOT:", V2_ROOT)
print("cwd:", Path.cwd())
print("smoke script:", (V2_ROOT / "scripts" / "smoke_generate.py").is_file())
print("experiment.yaml:", (V2_ROOT / "config" / "experiment.yaml").is_file())

MANUAL_V2_ROOT set but invalid: /content/drive/MyDrive/.../V2
Mounting Google Drive …


MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-usw1b0-3nfic6w3w75d8?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_lo

## 2. Install dependencies (Colab GPU)

In [ ]:
!pip -q install -r requirements.txt
# Preferred primary backend on Colab: CUDA llama-cpp (adjust CUDA wheel if needed)
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
# Fallback path uses transformers + bitsandbytes (already common on Colab):
# !pip -q install transformers accelerate bitsandbytes

## 3. Fingerprint + one generation smoke

Primary: `--backend llama_cpp`. Fallback: `--backend transformers`.

In [ ]:
!PYTHONPATH=. python scripts/smoke_generate.py --backend llama_cpp --notebook notebooks/colab_phase7_smoke.ipynb

In [ ]:
# Fallback if llama_cpp fails:
# !PYTHONPATH=. python scripts/smoke_generate.py --backend transformers

## 4. Confirm artefacts

Expected files:
- `results/config/phase7_runtime_fingerprint.json`
- `results/config/phase7_smoke_test.json` (validation evidence: PASS/FAIL + actual output)
- `project_record/evidence/phase7_validation.md` (update Colab row after run)

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase7_runtime_fingerprint.json')
smoke = Path('results/config/phase7_smoke_test.json')
print('fingerprint exists:', fp.is_file())
print('smoke_test exists:', smoke.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('backend:', (data.get('extra') or {}).get('backend_used'))
    print('actual:', repr(data.get('actual')))
    print('error:', data.get('error'))